# Actualización automática del DataFrame de criptomonedas

Este notebook descarga los días que faltan en `df_fear_greed_merged.csv` desde
la última fecha registrada hasta hoy, usando:

- **CoinGecko API** (gratis, sin key): precio, volumen, market cap diarios de BTC y ETH.
- **Alternative.me** (gratis, sin key): índice de miedo y codicia histórico completo.
- **CoinGecko /global**: dominancias actuales (solo del día de hoy, no histórico).

## Decisiones de diseño honestas

1. **Dominancias para días nuevos**: forward-fill desde el último valor del CSV.
   No calculamos dominancias rolling porque la API gratis no da `total_market_cap`
   histórico, y cualquier estimación rolling es una invención matemática.
2. **OHLC**: para días nuevos solo guardamos `close` y `volume`. La API gratuita
   de CoinGecko no devuelve OHLC fiable en el endpoint `/market_chart`. Para
   mantener coherencia con el CSV viejo, replicamos `close` en `open/high/low`,
   asumiendo que para análisis diario EOD esto es aceptable. Documenta esta
   limitación en tu TFG.
3. **Inflación y tipos de interés**: NO se actualizan aquí. Se añaden
   manualmente desde tus CSV mensuales en otro paso.
4. **NO se mergea con el CSV viejo en este notebook**. Solo se genera `df_nuevo`
   listo para concatenar.

## 1. Configuración

In [ ]:
import os
import time
import requests
import pandas as pd
from datetime import datetime

# Rutas
BASE_DIR        = r"C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF"
RUTA_CSV        = os.path.join(BASE_DIR, "data", "csv", "df_fear_greed_merged.csv")
RUTA_FG_RAW     = os.path.join(BASE_DIR, "data", "csv", "raw", "fear_greed_index.csv")
RUTA_DF_NUEVO   = os.path.join(BASE_DIR, "data", "csv", "raw", "df_dias_nuevos.csv")

# APIs
COINGECKO_BASE = "https://api.coingecko.com/api/v3"
ALTERNATIVE_BASE = "https://api.alternative.me/fng/"

# Rate limit (free tier CoinGecko: ~30 calls/min)
PAUSA_API = 2.5  # segundos entre llamadas

print("Configuración cargada ✓")
print(f"  CSV existente: {RUTA_CSV}")
print(f"  Salida nueva : {RUTA_DF_NUEVO}")

## 2. Diagnóstico: cuántos días faltan por actualizar

In [ ]:
df_merged = pd.read_csv(RUTA_CSV, parse_dates=["date"], index_col="date")
df_merged.sort_index(inplace=True)

ultima_fecha = df_merged.index.max().normalize()
hoy          = pd.Timestamp.now().normalize()
dias_faltan  = (hoy - ultima_fecha).days

print(f"Última fecha en CSV   : {ultima_fecha.date()}")
print(f"Fecha actual          : {hoy.date()}")
print(f"Días a descargar      : {dias_faltan}")
print(f"Columnas del CSV viejo: {df_merged.columns.tolist()}")

if dias_faltan <= 0:
    print("\n⚠️  El CSV ya está al día. No hay nada que descargar.")

## 3. Funciones de descarga

CoinGecko free tier solo acepta los rangos `1, 7, 14, 30, 90, 180, 365` días en `/market_chart`.
Pedimos el rango más pequeño que cubra `dias_faltan + margen`, y luego filtramos en pandas.

In [ ]:
def ajustar_dias_coingecko(dias_necesarios):
    """CoinGecko free tier solo acepta valores discretos."""
    valores_validos = [1, 7, 14, 30, 90, 180, 365]
    for v in valores_validos:
        if dias_necesarios <= v:
            return v
    return 365


def descargar_market_chart(coin_id, dias):
    """
    Descarga price + volume + market_cap diarios de CoinGecko.
    Devuelve DataFrame con columnas ['close', 'volume', 'market_cap'] indexado por fecha.
    """
    url = f"{COINGECKO_BASE}/coins/{coin_id}/market_chart"
    params = {"vs_currency": "usd", "days": dias, "interval": "daily"}
    
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"CoinGecko {coin_id} falló: {r.status_code} {r.text[:200]}")
    
    data = r.json()
    df_p = pd.DataFrame(data["prices"],        columns=["ts", "close"])
    df_v = pd.DataFrame(data["total_volumes"], columns=["ts", "volume"])
    df_m = pd.DataFrame(data["market_caps"],   columns=["ts", "market_cap"])
    
    for d in (df_p, df_v, df_m):
        d["date"] = pd.to_datetime(d["ts"], unit="ms").dt.normalize()
        d.drop(columns="ts", inplace=True)
        d.set_index("date", inplace=True)
    
    df = df_p.join(df_v).join(df_m)
    # Deduplicar por si hay varios timestamps por día
    df = df.groupby(df.index).last()
    return df


def descargar_dominancias_actuales():
    """
    /global solo da dominancias del momento actual (no histórico).
    Devuelve dict con btc_dominance, eth_dominance, alt_dominance (en tanto por uno).
    """
    r = requests.get(f"{COINGECKO_BASE}/global", timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"CoinGecko /global falló: {r.status_code}")
    
    g = r.json()["data"]["market_cap_percentage"]
    btc = g.get("btc", 0) / 100
    eth = g.get("eth", 0) / 100
    alt = 1 - btc - eth
    return {"btc_dominance": btc, "eth_dominance": eth, "alt_dominance": alt}


def descargar_fear_greed_completo():
    """
    Descarga el histórico completo del Fear & Greed Index de Alternative.me.
    """
    r = requests.get(ALTERNATIVE_BASE, params={"limit": 0, "format": "json"}, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Alternative.me falló: {r.status_code}")
    
    data = r.json()["data"]
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["timestamp"].astype(int), unit="s").dt.normalize()
    df["fear_greed"] = df["value"].astype(int)
    df["FearGreed_Label"] = df["value_classification"]
    return df[["date", "fear_greed", "FearGreed_Label"]].set_index("date").sort_index()


print("Funciones de descarga definidas ✓")

## 4. Descarga de los datos nuevos

In [ ]:
if dias_faltan <= 0:
    raise SystemExit("CSV ya actualizado.")

# Margen extra para asegurar overlap (los timestamps de CoinGecko no siempre cuadran)
dias_validos = ajustar_dias_coingecko(dias_faltan + 2)
print(f"Solicitando {dias_validos} días a CoinGecko (cubre los {dias_faltan} faltantes + margen)")

# Bitcoin
print("\n→ Bitcoin...")
df_btc = descargar_market_chart("bitcoin", dias_validos)
df_btc.columns = ["btc_close", "btc_volume", "btc_mcap"]
print(f"  {len(df_btc)} filas, rango {df_btc.index.min().date()} → {df_btc.index.max().date()}")
time.sleep(PAUSA_API)

# Ethereum
print("\n→ Ethereum...")
df_eth = descargar_market_chart("ethereum", dias_validos)
df_eth.columns = ["eth_close", "eth_volume", "eth_mcap"]
print(f"  {len(df_eth)} filas, rango {df_eth.index.min().date()} → {df_eth.index.max().date()}")
time.sleep(PAUSA_API)

df_nuevo = df_btc.join(df_eth, how="outer")
print(f"\nMerge BTC+ETH: {df_nuevo.shape}")

In [ ]:
# Dominancias del día actual (el endpoint /global solo da el momento presente)
print("→ Dominancias actuales (CoinGecko /global)...")
dom_actual = descargar_dominancias_actuales()
for k, v in dom_actual.items():
    print(f"  {k}: {v*100:.2f}%")
time.sleep(PAUSA_API)

In [ ]:
# Fear & Greed completo (lo descargamos entero, es ligero)
print("→ Fear & Greed (histórico completo)...")
df_fg = descargar_fear_greed_completo()
print(f"  {len(df_fg)} filas, rango {df_fg.index.min().date()} → {df_fg.index.max().date()}")

# Guardar el histórico raw por si hace falta más adelante
os.makedirs(os.path.dirname(RUTA_FG_RAW), exist_ok=True)
df_fg.to_csv(RUTA_FG_RAW)
print(f"  Guardado raw en: {RUTA_FG_RAW}")

## 5. Consolidar el DataFrame nuevo

Aquí componemos las filas que faltan en el CSV viejo. Para cada día nuevo:

- **OHLC**: replicamos `close` en open/high/low (limitación honesta de la API gratuita).
- **Dominancias**: forward-fill desde el último valor real del CSV viejo. No
  inventamos rolling: si actualizas a diario el sesgo es despreciable.
- **Fear & Greed**: del histórico de Alternative.me.
- **Inflación y tipos**: forward-fill del último valor del CSV viejo. Tendrás
  que actualizarlos manualmente cuando publiquen los datos mensuales.

In [ ]:
# 1. Quedarnos solo con los días posteriores a la última fecha del CSV viejo
df_nuevo = df_nuevo[df_nuevo.index > ultima_fecha].copy()
print(f"Días estrictamente nuevos tras filtrar: {len(df_nuevo)}")

if df_nuevo.empty:
    raise SystemExit("No hay días nuevos tras filtrar (margen consumido).")

# 2. Crear OHLC desde close (open=high=low=close)
for activo in ("btc", "eth"):
    df_nuevo[f"{activo}_open"] = df_nuevo[f"{activo}_close"]
    df_nuevo[f"{activo}_high"] = df_nuevo[f"{activo}_close"]
    df_nuevo[f"{activo}_low"]  = df_nuevo[f"{activo}_close"]

# 3. Añadir Fear & Greed
df_nuevo = df_nuevo.join(df_fg, how="left")

# 4. Dominancias: usar el valor actual de la API solo para el último día,
#    y forward-fill el resto desde el último valor del CSV viejo
ultima_dom = {
    "btc_dominance": df_merged["btc_dominance"].iloc[-1],
    "eth_dominance": df_merged["eth_dominance"].iloc[-1],
    "alt_dominance": df_merged["alt_dominance"].iloc[-1],
}
for k, v in ultima_dom.items():
    df_nuevo[k] = v

# Sobrescribir el último día con la dominancia real de la API
fecha_hoy = df_nuevo.index.max()
for k, v in dom_actual.items():
    df_nuevo.loc[fecha_hoy, k] = v

# 5. Inflación y fed_rate: forward-fill desde último valor del CSV viejo
if "inflation" in df_merged.columns:
    df_nuevo["inflation"] = df_merged["inflation"].iloc[-1]
if "fed_rate" in df_merged.columns:
    df_nuevo["fed_rate"] = df_merged["fed_rate"].iloc[-1]

# 6. Reordenar columnas igual que el CSV viejo
columnas_orden = df_merged.columns.tolist()
columnas_disponibles = [c for c in columnas_orden if c in df_nuevo.columns]
df_nuevo = df_nuevo[columnas_disponibles]

df_nuevo.index.name = "date"

print(f"\nDataFrame nuevo final: {df_nuevo.shape}")
print(f"Rango: {df_nuevo.index.min().date()} → {df_nuevo.index.max().date()}")
print(f"\nColumnas: {df_nuevo.columns.tolist()}")
print(f"\nNulos por columna:")
print(df_nuevo.isna().sum()[df_nuevo.isna().sum() > 0])
df_nuevo

## 6. Guardar el DataFrame nuevo

Lo guardamos aparte. NO se mergea con el CSV viejo aquí; eso lo haces tú
cuando hayas validado que las filas son correctas, así no corrompes el
histórico por error.

In [ ]:
os.makedirs(os.path.dirname(RUTA_DF_NUEVO), exist_ok=True)
df_nuevo.to_csv(RUTA_DF_NUEVO)
print(f"✓ Guardado en: {RUTA_DF_NUEVO}")
print(f"  Filas: {len(df_nuevo)}")
print(f"  Columnas: {len(df_nuevo.columns)}")

## 7. (Opcional) Merge manual con el CSV viejo

Cuando hayas revisado `df_nuevo` y veas que está OK, descomenta y ejecuta
esta celda para mergear y sobrescribir el CSV principal.

**Antes de ejecutar:** revisa que las dominancias forward-filled tengan
sentido (no salten de golpe), y que `inflation` y `fed_rate` sigan a niveles
razonables hasta que los actualices manualmente con el dato mensual real.

In [ ]:
# DESCOMENTA SOLO CUANDO HAYAS VALIDADO df_nuevo

# df_actualizado = pd.concat([df_merged, df_nuevo])
# df_actualizado = df_actualizado[~df_actualizado.index.duplicated(keep='last')]
# df_actualizado.sort_index(inplace=True)
# 
# # Backup del CSV viejo antes de sobrescribir
# import shutil
# backup = RUTA_CSV.replace('.csv', f'_backup_{hoy.strftime("%Y%m%d")}.csv')
# shutil.copy(RUTA_CSV, backup)
# print(f'Backup guardado: {backup}')
# 
# df_actualizado.to_csv(RUTA_CSV)
# print(f'CSV principal actualizado: {len(df_actualizado)} filas totales')
# print(f'Última fecha: {df_actualizado.index.max().date()}')